© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# HAIO 2024 - Spectrogram Megoldas

Zene-klasszifikacio a GTZAN adathalmaz spektrogramjai alapjan. 10 zenei mufaj, mufajonkent 100 minta.

---
## Elokeszuletek

In [ ]:
# Installing the required packages
!sudo apt-get install -y ffmpeg --quiet
!pip install librosa timm --quiet

In [ ]:
import os
import glob
import random
import shutil
import requests
import numpy as np
import matplotlib.pyplot as plt
from zipfile import ZipFile

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

import librosa
import librosa.display
import IPython.display as display

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

### Adathalmaz letoltese

In [ ]:
# Downloading and extracting the GTZAN dataset
fname = "music.zip"
url = "https://osf.io/drjhb/download"

if not os.path.isfile(fname):
    try:
        r = requests.get(url)
    except requests.ConnectionError:
        print("!!! Failed to download the dataset !!!")
    else:
        if r.status_code != requests.codes.ok:
            print("!!! Failed to download the dataset !!!")
        else:
            with open(fname, "wb") as fid:
                fid.write(r.content)

with ZipFile(fname, 'r') as zipObj:
    zipObj.extractall()

print("Extraction complete!")

In [ ]:
# Train/test split: 80 training / 20 test per genre
spectrograms_dir = "Data/images_original/"
folder_names = ['Data/train/', 'Data/test/']
train_dir = folder_names[0]
test_dir = folder_names[1]

for f in folder_names:
    if os.path.exists(f):
        shutil.rmtree(f)
        os.mkdir(f)
    else:
        os.mkdir(f)

genres = sorted(os.listdir(spectrograms_dir))
print(f'Genres: {genres}')

for g in genres:
    src_file_paths = []
    for im in glob.glob(os.path.join(spectrograms_dir, f'{g}', '*.png'), recursive=True):
        src_file_paths.append(im)
    random.shuffle(src_file_paths)
    test_files = src_file_paths[0:20]
    train_files = src_file_paths[20:]

    for f in folder_names:
        if not os.path.exists(os.path.join(f + f'{g}')):
            os.mkdir(os.path.join(f + f'{g}'))

    for f in train_files:
        shutil.copy(f, os.path.join(os.path.join(train_dir + f'{g}') + '/', os.path.split(f)[1]))
    for f in test_files:
        shutil.copy(f, os.path.join(os.path.join(test_dir + f'{g}') + '/', os.path.split(f)[1]))

print(f'Training samples: {sum(len(os.listdir(os.path.join(train_dir, g))) for g in genres)}')
print(f'Test samples: {sum(len(os.listdir(os.path.join(test_dir, g))) for g in genres)}')

In [ ]:
# Loading the data with ImageFolder
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=25, shuffle=False, num_workers=0)

class_names = train_dataset.classes
print(f'Classes: {class_names}')
print(f'Number of training samples: {len(train_dataset)}, Number of test samples: {len(test_dataset)}')

In [ ]:
# Helper functions for training and evaluation

def plot_loss_accuracy(train_loss, train_acc, validation_loss, validation_acc):
    """Plot the training and validation loss/accuracy curves."""
    epochs = len(train_loss)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(range(epochs), train_loss, label='Training Loss')
    ax1.plot(range(epochs), validation_loss, label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Epoch vs Loss')
    ax1.legend()

    ax2.plot(range(epochs), train_acc, label='Training Accuracy')
    ax2.plot(range(epochs), validation_acc, label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Epoch vs Accuracy')
    ax2.legend()
    plt.tight_layout()
    plt.show()


def evaluate(model, data_loader, device):
    """Evaluate the model on a given dataset."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return correct / total


def train_model(model, device, train_loader, test_loader, epochs, lr=0.001):
    """Generic training loop with loss and accuracy tracking."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_loss_hist, val_loss_hist = [], []
    train_acc_hist, val_acc_hist = [], []

    for epoch in tqdm(range(epochs), desc='Training'):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

        train_loss_hist.append(running_loss / len(train_loader))
        train_acc_hist.append(correct / total)

        # Validation
        model.eval()
        running_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                running_loss += loss.item()
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        val_loss_hist.append(running_loss / len(test_loader))
        val_acc_hist.append(correct / total)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} - '
                  f'Train Loss: {train_loss_hist[-1]:.4f}, Train Acc: {train_acc_hist[-1]:.4f}, '
                  f'Val Loss: {val_loss_hist[-1]:.4f}, Val Acc: {val_acc_hist[-1]:.4f}')

    return train_loss_hist, train_acc_hist, val_loss_hist, val_acc_hist

---
# Task 1 - Data visualization

Let's examine whether the genres can be distinguished based on the spectrograms: display one spectrogram per genre on a shared figure.

In [ ]:
# Displaying one spectrogram per genre
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Spectrograms by genre', fontsize=16)

for idx, genre in enumerate(class_names):
    ax = axes[idx // 5, idx % 5]
    # Load the first image from the given genre
    genre_dir = os.path.join(spectrograms_dir, genre)
    img_files = sorted(os.listdir(genre_dir))
    img_path = os.path.join(genre_dir, img_files[0])
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.set_title(genre, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('The spectrograms show that the different genres exhibit distinct patterns.')
print('For example, the classical genre is less intense at higher frequencies,')
print('while the metal/rock genres show a more intense, broader-band pattern.')

---
# Task 2 - CNN classification

We train a convolutional network from scratch (5 convolutional layers, batch normalization, dropout, max pooling). Goal: >75% test accuracy.

**Settings:**
- Resize images to 224x224
- Adam optimizer, lr=0.001
- CrossEntropyLoss
- 50 epochs
- Batch size: 25

In [ ]:
class MusicNet(nn.Module):
    """A network with 5 convolutional layers, batch normalization and dropout."""
    def __init__(self):
        super(MusicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=0)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=0)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=0)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=0)

        self.batchnorm1 = nn.BatchNorm2d(8)
        self.batchnorm2 = nn.BatchNorm2d(16)
        self.batchnorm3 = nn.BatchNorm2d(32)
        self.batchnorm4 = nn.BatchNorm2d(64)
        self.batchnorm5 = nn.BatchNorm2d(128)

        self.dropout = nn.Dropout(p=0.3)

        # Compute the number of inputs of the fully connected layer from the image size
        # 224 -> conv+pool -> 111 -> conv+pool -> 54 -> conv+pool -> 26 -> conv+pool -> 12 -> conv+pool -> 5
        # 128 * 5 * 5 = 3200
        self.fc1 = nn.Linear(128 * 5 * 5, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.batchnorm1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm3(self.conv3(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm4(self.conv4(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm5(self.conv5(x))), 2)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        return x

In [ ]:
# Training the CNN
cnn_model = MusicNet().to(device)
print(f'Number of parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

cnn_train_loss, cnn_train_acc, cnn_val_loss, cnn_val_acc = train_model(
    cnn_model, device, train_loader, test_loader, epochs=50, lr=0.001
)

plot_loss_accuracy(cnn_train_loss, cnn_train_acc, cnn_val_loss, cnn_val_acc)

# Final accuracy
cnn_test_acc = evaluate(cnn_model, test_loader, device)
print(f'\nCNN test accuracy: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')

---
# Task 3 - Augmentation (SpecAugment)

We apply the SpecAugment method proposed in Park et al. 2019:
- **Frequency masking**: removing random horizontal bands
- **Time masking**: removing random vertical bands

These methods preserve the temporal correlation, unlike traditional image augmentation.

In [ ]:
class FrequencyMasking:
    """Frequency masking: removing a random horizontal band from the spectrogram.
    Park et al., 2019 - SpecAugment
    """
    def __init__(self, freq_mask_param=20):
        self.freq_mask_param = freq_mask_param

    def __call__(self, img):
        # img: tensor [C, H, W]
        _, h, w = img.shape
        f = random.randint(0, min(self.freq_mask_param, h - 1))
        f0 = random.randint(0, h - f)
        img[:, f0:f0 + f, :] = 0
        return img


class TimeMasking:
    """Time masking: removing a random vertical band from the spectrogram.
    Park et al., 2019 - SpecAugment
    """
    def __init__(self, time_mask_param=20):
        self.time_mask_param = time_mask_param

    def __call__(self, img):
        # img: tensor [C, H, W]
        _, h, w = img.shape
        t = random.randint(0, min(self.time_mask_param, w - 1))
        t0 = random.randint(0, w - t)
        img[:, :, t0:t0 + t] = 0
        return img


# Augmented transformation
augmented_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    FrequencyMasking(freq_mask_param=20),
    TimeMasking(time_mask_param=20),
])

# Augmented dataset and loader
aug_train_dataset = datasets.ImageFolder(train_dir, transform=augmented_transform)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=25, shuffle=True, num_workers=0)

# Let's visualize the augmented images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Examples of augmented spectrograms', fontsize=14)
for i in range(4):
    img, label = aug_train_dataset[i * 20]
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    axes[i].set_title(class_names[label])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# The same CNN architecture, but trained on augmented data
cnn_aug_model = MusicNet().to(device)

aug_train_loss, aug_train_acc, aug_val_loss, aug_val_acc = train_model(
    cnn_aug_model, device, aug_train_loader, test_loader, epochs=50, lr=0.001
)

plot_loss_accuracy(aug_train_loss, aug_train_acc, aug_val_loss, aug_val_acc)

aug_test_acc = evaluate(cnn_aug_model, test_loader, device)
print(f'\nCNN (augmented) test accuracy: {aug_test_acc:.4f} ({aug_test_acc*100:.1f}%)')
print(f'CNN (original) test accuracy: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')
print(f'Difference: {(aug_test_acc - cnn_test_acc)*100:+.1f}%')

---
# 4. Feladat - Transzfer tanitas

ImageNet-en elotanitott DenseNet121 modell finomhangolasa a zene-klasszifikacios feladatra.
A Palanisamy et al. 2020 cikk alapjan a termeszetes kepeken elotanitott modellek sikeresen alkalmazhatok spektrogramokra is.

In [ ]:
# Loading DenseNet121 with ImageNet weights
densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

# Replacing the last layer with one for 10 classes
num_features = densenet.classifier.in_features
densenet.classifier = nn.Linear(num_features, 10)
densenet = densenet.to(device)

print(f'DenseNet121 - last layer input: {num_features}, output: 10')

# Applying ImageNet normalization
imagenet_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

imagenet_train_dataset = datasets.ImageFolder(train_dir, transform=imagenet_transform)
imagenet_test_dataset = datasets.ImageFolder(test_dir, transform=imagenet_transform)
imagenet_train_loader = DataLoader(imagenet_train_dataset, batch_size=25, shuffle=True, num_workers=0)
imagenet_test_loader = DataLoader(imagenet_test_dataset, batch_size=25, shuffle=False, num_workers=0)

In [ ]:
# Fine-tuning DenseNet
dn_train_loss, dn_train_acc, dn_val_loss, dn_val_acc = train_model(
    densenet, device, imagenet_train_loader, imagenet_test_loader, epochs=20, lr=0.0001
)

plot_loss_accuracy(dn_train_loss, dn_train_acc, dn_val_loss, dn_val_acc)

dn_test_acc = evaluate(densenet, imagenet_test_loader, device)
print(f'\nDenseNet121 test accuracy: {dn_test_acc:.4f} ({dn_test_acc*100:.1f}%)')
print(f'CNN (from scratch) test accuracy: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')
print(f'Transfer learning typically gives better results, because ImageNet pretraining')
print(f'has learned useful visual features that can also be applied to spectrograms.')

---
# 5. Feladat - Klaszterezes

A DenseNet modell utolso retege elotti jellemzovektorokat kinyerjuk, majd t-SNE-vel 2D-be vetitjuk es abrazoljuk mufaj szerint szinezve.

In [ ]:
# Extracting the feature vector from before DenseNet's last layer
# We use a hook to extract the input of the last layer

features_list = []
labels_list = []

# Registering a hook before the classifier layer
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = input[0].detach()
    return hook

hook_handle = densenet.classifier.register_forward_hook(get_activation('features'))

densenet.eval()
with torch.no_grad():
    for data, target in imagenet_test_loader:
        data = data.to(device)
        _ = densenet(data)
        features_list.append(activation['features'].cpu().numpy())
        labels_list.append(target.numpy())

    # Feature vectors of the training data as well (for clustering and similarity)
    train_features_list = []
    train_labels_list = []
    for data, target in imagenet_train_loader:
        data = data.to(device)
        _ = densenet(data)
        train_features_list.append(activation['features'].cpu().numpy())
        train_labels_list.append(target.numpy())

hook_handle.remove()

# Concatenating all feature vectors
all_features = np.concatenate(features_list + train_features_list, axis=0)
all_labels = np.concatenate(labels_list + train_labels_list, axis=0)

test_features = np.concatenate(features_list, axis=0)
test_labels = np.concatenate(labels_list, axis=0)

print(f'Feature vector size: {all_features.shape[1]}')
print(f'Total samples: {all_features.shape[0]}')

In [ ]:
# t-SNE projection and visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(all_features)

plt.figure(figsize=(12, 10))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, genre in enumerate(class_names):
    mask = all_labels == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
                c=[colors[i]], label=genre, alpha=0.7, s=30)

plt.legend(fontsize=12)
plt.title('t-SNE projection colored by genre', fontsize=16)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

print('The visualization shows that the genres largely form separate clusters.')
print('Some genres (e.g. rock and metal) lie closer to each other, which reflects their musical similarity.')

---
# 6. Feladat - Hasonlosag

Kiszamitjuk a koszinusz hasonlosagot a mufajok kozotti jellemzovektor-centroidok (atlagvektorok) kozott, es heatmap formajaban abrazoljuk.

In [ ]:
# Computing per-genre centroids (mean feature vector)
centroids = []
for i in range(10):
    mask = all_labels == i
    centroid = all_features[mask].mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(f'Shape of the centroids: {centroids.shape}')

# Cosine similarity matrix
cos_sim_matrix = cosine_similarity(centroids)

# Displaying the heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cos_sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_title('Cosine similarity between genres', fontsize=16)

# Writing the values into the cells
for i in range(10):
    for j in range(10):
        text = ax.text(j, i, f'{cos_sim_matrix[i, j]:.2f}',
                       ha='center', va='center', fontsize=8,
                       color='white' if abs(cos_sim_matrix[i, j]) > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.show()

print('The heatmap shows which genres are most similar to one another.')
print('A higher value indicates greater similarity based on the feature spaces.')

---
# Task 7 - Integrated gradients

Based on the paper by Sundararajan et al. 2017, let's implement the integrated gradients method,
which shows which pixels the model considers important for its decision.

In [ ]:
def integrated_gradients(model, input_tensor, target_class, baseline=None, steps=50):
    """Computing integrated gradients (Sundararajan et al., 2017).
    
    The integrated gradient integrates the gradients along the straight path leading
    from the baseline (default: black image) to the input.
    
    IG(x) = (x - x') * integral_0^1 dF(x' + a*(x-x'))/dx da
    """
    if baseline is None:
        baseline = torch.zeros_like(input_tensor)
    
    # Generating the interpolation steps
    scaled_inputs = [baseline + (float(i) / steps) * (input_tensor - baseline) 
                     for i in range(steps + 1)]
    
    # Computing the gradients at every step
    grads = []
    model.eval()
    for scaled_input in scaled_inputs:
        scaled_input = scaled_input.to(device).requires_grad_(True)
        output = model(scaled_input)
        output[0, target_class].backward()
        grads.append(scaled_input.grad.detach().cpu())
    
    # Numerical integration using the trapezoidal rule
    grads = torch.stack(grads)
    avg_grads = (grads[:-1] + grads[1:]).mean(dim=0)
    
    # Integrated gradient = (input - baseline) * average gradient
    integrated_grad = (input_tensor.cpu() - baseline.cpu()) * avg_grads
    
    return integrated_grad

In [ ]:
# Let's choose a test sample
test_iter = iter(imagenet_test_loader)
test_images, test_targets = next(test_iter)

sample_idx = 0
sample_image = test_images[sample_idx:sample_idx+1]
sample_target = test_targets[sample_idx].item()

print(f'Genre of the selected sample: {class_names[sample_target]}')

# Computing the integrated gradients
ig = integrated_gradients(densenet, sample_image, sample_target, steps=50)

# Visualization
# The attribution map: summing over the channels and taking the absolute value
attribution = ig.squeeze(0).sum(dim=0).numpy()  # [H, W]
attribution = np.abs(attribution)
# Normalization
attribution = attribution / (attribution.max() + 1e-8)

# Denormalizing the original image for display
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
orig_img = sample_image.squeeze(0).permute(1, 2, 0).numpy()
orig_img = orig_img * std + mean
orig_img = np.clip(orig_img, 0, 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.imshow(orig_img)
ax1.set_title(f'Original spectrogram ({class_names[sample_target]})', fontsize=13)
ax1.axis('off')

ax2.imshow(attribution, cmap='hot')
ax2.set_title('Integrated gradients', fontsize=13)
ax2.axis('off')

ax3.imshow(orig_img)
ax3.imshow(attribution, cmap='hot', alpha=0.5)
ax3.set_title('Attribution overlay', fontsize=13)
ax3.axis('off')

plt.suptitle('Integrated gradients visualization', fontsize=16)
plt.tight_layout()
plt.show()

print('The integrated gradients show which frequency ranges and which parts in time')
print('the model uses most for identifying the genre.')

---
# 8. Feladat - Vision Transformer

Elotanitott Vision Transformer (ViT) modell finomhangolasa a spektrogram-klasszifikaciora.
Osszehasonlitjuk a CNN es a transzfer tanulas eredmenyeivel.

In [ ]:
import timm

# Loading the ViT model with pretrained weights
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)
vit_model = vit_model.to(device)

print(f'Number of ViT parameters: {sum(p.numel() for p in vit_model.parameters()):,}')

In [ ]:
# Fine-tuning the ViT
# We use a lower learning rate for the large pretrained model
vit_train_loss, vit_train_acc, vit_val_loss, vit_val_acc = train_model(
    vit_model, device, imagenet_train_loader, imagenet_test_loader, epochs=15, lr=0.00005
)

plot_loss_accuracy(vit_train_loss, vit_train_acc, vit_val_loss, vit_val_acc)

vit_test_acc = evaluate(vit_model, imagenet_test_loader, device)
print(f'\n--- Comparison ---')
print(f'CNN (from scratch):  {cnn_test_acc*100:.1f}%')
print(f'CNN (augmented):     {aug_test_acc*100:.1f}%')
print(f'DenseNet121 (tl):    {dn_test_acc*100:.1f}%')
print(f'Vision Transformer:  {vit_test_acc*100:.1f}%')

---
# 9. Feladat - Generalas

Variational Autoencoder (VAE) tanitasa a spektrogramokon, uj spektrogramok generalasa,
majd a generalt kepek visszaalakitasa audiova a Griffin-Lim algoritmussal.

In [ ]:
# VAE architecture
class VAE(nn.Module):
    """Variational Autoencoder for spectrograms."""
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # 224 -> 112
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 112 -> 56
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 56 -> 28
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), # 28 -> 14
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        
        # Latent space: mu and logvar
        self.fc_mu = nn.Linear(256 * 14 * 14, latent_dim)
        self.fc_logvar = nn.Linear(256 * 14 * 14, latent_dim)
        
        # Decoder
        self.fc_decode = nn.Linear(latent_dim, 256 * 14 * 14)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # 14 -> 28
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 28 -> 56
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # 56 -> 112
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # 112 -> 224
            nn.Sigmoid(),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + sigma * epsilon"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(h.size(0), 256, 14, 14)
        return self.decoder(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(recon_x, x, mu, logvar):
    """VAE loss function: reconstruction error + KL divergence."""
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

In [ ]:
# Training the VAE
vae = VAE(latent_dim=128).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=0.001)

# Simpler loader (without normalization, because the VAE reconstructs pixel values)
vae_train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)

vae_losses = []
num_epochs = 50

for epoch in tqdm(range(num_epochs), desc='VAE training'):
    vae.train()
    epoch_loss = 0
    for data, _ in vae_train_loader:
        data = data.to(device)
        vae_optimizer.zero_grad()
        recon, mu, logvar = vae(data)
        loss = vae_loss(recon, data, mu, logvar)
        loss.backward()
        vae_optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(vae_train_loader.dataset)
    vae_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{num_epochs}, Average loss: {avg_loss:.2f}')

plt.figure(figsize=(10, 4))
plt.plot(vae_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE training loss')
plt.show()

In [ ]:
# Generating new spectrograms from the latent space
vae.eval()
with torch.no_grad():
    # Sampling random latent vectors
    z = torch.randn(8, 128).to(device)
    generated = vae.decode(z).cpu()

# Displaying the generated and reconstructed images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Generated spectrograms', fontsize=16)

for i in range(8):
    ax = axes[i // 4, i % 4]
    img = generated[i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'Generated #{i+1}')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Displaying the reconstructions: original vs reconstructed
test_batch, _ = next(iter(DataLoader(test_dataset, batch_size=4, shuffle=True)))
with torch.no_grad():
    recon, _, _ = vae(test_batch.to(device))
    recon = recon.cpu()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Original vs Reconstructed', fontsize=16)
for i in range(4):
    axes[0, i].imshow(test_batch[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(recon[i].permute(1, 2, 0).numpy(), 0, 1))
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Converting a generated spectrogram back to audio with the Griffin-Lim algorithm

def spectrogram_image_to_audio(img_array, sr=22050, n_fft=2048, hop_length=512, n_mels=128):
    """Converting a spectrogram image back to audio.
    
    1. Converting the image to grayscale
    2. Resizing it to the mel-spectrogram size
    3. Mel-spectrogram -> linear spectrogram inversion
    4. Griffin-Lim phase reconstruction
    """
    # Grayscale conversion
    if img_array.ndim == 3:
        gray = np.mean(img_array, axis=2)
    else:
        gray = img_array
    
    # Resizing to the mel-spectrogram size
    from PIL import Image
    img_pil = Image.fromarray((gray * 255).astype(np.uint8))
    img_resized = img_pil.resize((431, n_mels))  # typical GTZAN spectrogram size
    mel_spec = np.array(img_resized).astype(np.float32) / 255.0
    
    # Scaling: pixel values -> dB
    mel_spec_db = mel_spec * 80 - 80  # approximate dB range
    
    # dB -> amplitude
    mel_spec_power = librosa.db_to_power(mel_spec_db)
    
    # Mel -> linear frequency inversion
    mel_basis = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels)
    # Pseudo-inverse of the mel filter bank
    mel_basis_pinv = np.linalg.pinv(mel_basis)
    spec = np.maximum(0, mel_basis_pinv @ mel_spec_power)
    
    # Griffin-Lim phase reconstruction
    audio = librosa.griffinlim(spec, hop_length=hop_length, n_iter=64)
    
    return audio


# Converting the generated spectrograms to audio
print('Converting the generated spectrograms back to audio...')
print('='*50)

for i in range(min(4, generated.shape[0])):
    img = generated[i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    audio = spectrogram_image_to_audio(img)
    
    # Normalizing the audio
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    
    print(f'\nGenerated music #{i+1} (length: {len(audio)/22050:.1f} seconds):')
    display.display(display.Audio(audio, rate=22050))

print('\nNote: The quality of the spectrograms generated by the VAE and of the sounds')
print('produced from them is limited, since the model was trained on relatively little data.')
print('With more complex architectures (e.g. GAN, diffusion model) better quality can be achieved.')

---
## Summary

| Model | Test accuracy |
|--------|----------------|
| CNN (from scratch) | see above |
| CNN + SpecAugment | see above |
| DenseNet121 (transfer) | see above |
| Vision Transformer | see above |

Transfer learning and the Vision Transformer generally achieved better results
than the CNN trained from scratch, which shows that the visual features learned
on ImageNet are also useful for spectrogram classification.